## Multi-modal prompting

#### Basic prompting demo with Llama 3.2

In [ ]:
import ollama
import yfinance as yf
from PIL import Image
from IPython.display import Markdown, display

In [ ]:
response1 = ollama.chat(
    model='llama3.2-vision',
    messages = [{
        "role": "user",
         "content": "Who wrote the book Lord of the Rings?"
         }]
)

display(Markdown(response1.message.content))

#### Supported Roles

**System**  

Defines the context for interacting with AI. Typically, it includes rules and guidelines to help the model respond effectively.  
*Example:* "You are a helpful assistant skilled in explaining complex topics in simple terms."

**User**  

Represents the human input to the model, including questions, commands, and prompts directed to the AI.  
*Example:* "Can you explain the concept of quantum entanglement in simple words?"

**Assistant**  

Represents the responses generated by the AI, based on the context provided by the system or user prompts.  
*Example:* "Quantum entanglement is a phenomenon where two particles remain interconnected, so the state of one instantly influences the state of the other, no matter how far apart they are."

### Here's another example

In [ ]:
response2 = ollama.chat(
    model='llama3.2-vision',    
    messages = [
      {"role": "user",
       "content": "Who wrote the book Lord of the Rings?"},
      {"role": "assistant",
       "content": response1.message.content},
      {"role": "user",
       "content": "What other books has the author written?"}
    ],
)
display(Markdown(response2.message.content))

## Let's organise what we've done so far

In [ ]:
class Conversation:
    def __init__(self, system=""):
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})
    def generate(self, user_question):
        self.messages.append({"role": "user", "content":user_question})
        response = ollama.chat(model='llama3.2-vision', messages=self.messages)
        self.messages.append({"role":"assistant", "content":response.message.content})
        return response

In [ ]:
system_message = "You are a terse expert in high fantasy literature." 
conv = Conversation(system_message)
response = conv.generate("Who wrote the book Lord of the Rings?")
display(Markdown(response2.message.content))

In [ ]:
conv.messages

### Multimodal prompting

In [ ]:
# We'll ask the model to describe this image
Image.open('dogs.png')

In [ ]:

response = ollama.chat(
    model='llama3.2-vision',
    messages=[{
        'role': 'user',
        'content': 'Describe this image.',
        'images': ['dogs.png']
    }]
)

display(Markdown(response.message.content))

#### Multiple images

In [ ]:

response = ollama.chat(
    model='llama3.2-vision',
    messages=[{
        'role': 'user',
        'content': 'Calculate the total bill amount.',
        'images': ['receipt.png', 'receipt.jpg']
    }]
)

display(Markdown(response.message.content))

## OCR demo

In [ ]:
# Sample restaurant receipt
Image.open('receipt.jpg')

In [ ]:
response = ollama.chat(
    model='llama3.2-vision',
    messages=[{
        'role': 'user',
        'content': """Analyze the text in the provided image. Extract all readable content
                      and present it in a structured Markdown format that is clear, concise, 
                      and well-organized. Ensure proper formatting (e.g., headings, lists, or
                      code blocks) as necessary to represent the content effectively.""",
        'images': ['receipt.jpg']
    }]
)

display(Markdown(response.message.content))

#### Tool calling demo

In [ ]:
# ! pip install yfinance

In [ ]:
import ollama
import yfinance as yf

prompt = "What is the stock price of Apple?"

# define a function to use as tool
def get_stock_price(ticker: str) -> float:
    stock = yf.Ticker(ticker)
    return stock.history(period='1d')['Close'].iloc[-1]

# Pass the function as a tool to Ollama
response = ollama.chat(
    'llama3.2',
    messages=[{'role': 'user', 'content': 'What is the stock price of Apple?'}],
    tools=[get_stock_price],  # Actual function reference
)

# Call the function from the model response
available_functions = {
    'get_stock_price': get_stock_price,
}

for tool in response.message.tool_calls or []:
    function_to_call = available_functions.get(tool.function.name)
    if function_to_call:
        print('Agruments:', tool.function.arguments)
        print('Function output:', function_to_call(**tool.function.arguments))
    else:
        print('Function not found:', tool.function.name)

In [ ]:
import ollama
import yfinance as yf

prompt = "What is the stock price of Apple?"

# define a function to use as tool
def get_stock_price(ticker: str) -> float:
    stock = yf.Ticker(ticker)
    return stock.history(period='1d')['Close'].iloc[-1]

# Pass the function as a tool to Ollama
response = ollama.chat(
    'llama3.2',
    messages=[{'role': 'user', 'content': 'What is the stock price of Apple?'}],
    tools=[get_stock_price],  # Actual function reference
)

# Call the function from the model response
available_functions = {
    'get_stock_price': get_stock_price,
}

for tool in response.message.tool_calls or []:
    function_to_call = available_functions.get(tool.function.name)
    if function_to_call:
        print('Agruments:', tool.function.arguments)
        print('Function output:', function_to_call(**tool.function.arguments))
    else:
        print('Function not found:', tool.function.name)